# Merge all datasets

Run the code cell below. Rent and income receive a `postalcode` derived from their INSEE commune code before merging.

In [11]:
from io import BytesIO
from pathlib import Path
import re
import struct
import unicodedata
import zipfile

import pandas as pd
import requests

SOURCES = {
    'postal_codes': 'https://www.data.gouv.fr/api/1/datasets/r/c63fd0b1-7987-46f6-b779-8b3ed889090c',
    'rent': 'https://www.data.gouv.fr/api/1/datasets/r/55b34088-0964-415f-9df7-d87dd98a09be',
    'income': 'https://www.data.gouv.fr/api/1/datasets/r/6d7542dd-4aa7-4683-928e-df4292f5cd53',
    'libraries': 'https://www.data.gouv.fr/api/1/datasets/r/806a8aa1-952f-404d-9857-3f27b7c0ca86',
    'bookshops': 'https://www.data.gouv.fr/api/1/datasets/r/bcd7d584-89c3-46b6-a164-4582eb2d43b5',
    'population_history': 'https://www.data.gouv.fr/api/1/datasets/r/be303501-5c46-48a1-87b4-3d198423ff49',
}

def normalise_name(name):
    text = unicodedata.normalize('NFKD', str(name))
    text = ''.join(char for char in text if not unicodedata.combining(char))
    return re.sub(r'[^a-z0-9]', '', text.lower())

def clean_postalcode(values):
    return (values.astype('string').str.strip().str.replace(r'\.0$', '', regex=True)
            .str.zfill(5).where(values.notna()))

def clean_insee_code(values):
    return (values.astype('string').str.strip().str.replace(r'^COM-', '', regex=True)
            .str.replace(r'\.0$', '', regex=True).str.zfill(5).where(values.notna()))

def postal_column(df):
    names = {'postalcode', 'postcode', 'zipcode', 'zip', 'codepostal', 'cp'}
    return next((col for col in df.columns if normalise_name(col) in names), None)

def insee_column(df):
    names = {'codeinsee', 'inseec', 'codegeog', 'codgeo', 'codecommune', 'codecom'}
    return next((col for col in df.columns if normalise_name(col) in names), None)

def read_dbf(raw):
    record_count = struct.unpack('<I', raw[4:8])[0]
    header_length = struct.unpack('<H', raw[8:10])[0]
    record_length = struct.unpack('<H', raw[10:12])[0]
    fields, position = [], 32
    while raw[position] != 13:
        field = raw[position:position + 32]
        raw_name = field[:11].split(b'\0')[0]
        try:
            field_name = raw_name.decode('utf-8')
        except UnicodeDecodeError:
            field_name = raw_name.decode('cp1252')
        fields.append((field_name, chr(field[11]), field[16], field[17]))
        position += 32

    rows = []
    for row_number in range(record_count):
        start = header_length + row_number * record_length
        record = raw[start:start + record_length]
        if not record or record[0] == 42:
            continue
        row, offset = {}, 1
        for field_name, field_type, width, decimals in fields:
            value = record[offset:offset + width].decode('cp1252', errors='replace').strip()
            offset += width
            if not value or set(value) == {'*'}:
                row[field_name] = pd.NA
            elif field_type in 'NF':
                row[field_name] = float(value) if decimals else int(value)
            else:
                row[field_name] = value
        rows.append(row)
    return pd.DataFrame(rows)

def download_table(url):
    response = requests.get(url, timeout=120)
    response.raise_for_status()
    data = BytesIO(response.content)
    content_type = response.headers.get('Content-Type', '').lower()

    if zipfile.is_zipfile(data):
        data.seek(0)
        with zipfile.ZipFile(data) as archive:
            dbf_files = [name for name in archive.namelist() if name.lower().endswith('.dbf')]
            if dbf_files:
                return read_dbf(archive.read(dbf_files[0]))
        data.seek(0)

    if 'excel' in content_type or response.content[:2] == b'PK':
        return pd.read_excel(data)
    if 'json' in content_type:
        return pd.read_json(data)

    for encoding in ('utf-8', 'cp1252', 'latin1'):
        try:
            data.seek(0)
            return pd.read_csv(data, sep=None, engine='python', encoding=encoding)
        except UnicodeDecodeError:
            pass
    raise ValueError('Unable to decode CSV data.')

tables = {name: download_table(url) for name, url in SOURCES.items()}
for name, table in tables.items():
    print(f'Downloaded {name}: {len(table):,} rows × {len(table.columns)} columns')

big_dataset = tables['postal_codes'].copy()
base_postal = postal_column(big_dataset)
base_insee = insee_column(big_dataset)
if base_postal is None or base_insee is None:
    raise ValueError('The first dataset must contain postalcode and code_insee columns.')

big_dataset = big_dataset.rename(columns={base_postal: 'postalcode'})
big_dataset['postalcode'] = clean_postalcode(big_dataset['postalcode'])
insee_to_postal = (
    big_dataset.assign(_insee=clean_insee_code(big_dataset[base_insee]))
    .dropna(subset=['_insee', 'postalcode'])
    .drop_duplicates('_insee')
    .set_index('_insee')['postalcode']
)

for name, table in tables.items():
    if name == 'postal_codes':
        continue
    table = table.copy()
    if name == 'population_history':
        table = table.drop(columns=[f'p{year}_pop' for year in range(13, 23)], errors='ignore')
    key = postal_column(table)

    if key is not None:
        table = table.rename(columns={key: 'postalcode'})
        table['postalcode'] = clean_postalcode(table['postalcode'])
    else:
        key = insee_column(table)
        if key is None:
            raise ValueError(f'{name} has no postal-code or INSEE commune-code column.')
        table['postalcode'] = clean_insee_code(table[key]).map(insee_to_postal)
        print(f'{name}: postalcode derived from {key}; {table.postalcode.isna().sum():,} unmatched rows kept.')

    table = table.rename(columns={col: f'{name}_{col}' for col in table.columns if col != 'postalcode'})
    table = table.groupby('postalcode', dropna=False).first().reset_index()
    big_dataset = big_dataset.merge(table, on='postalcode', how='left')

big_dataset.to_csv('big_merged_dataset.csv', index=False, encoding='utf-8-sig')
print(f'Created big_merged_dataset.csv: {len(big_dataset):,} rows × {len(big_dataset.columns)} columns')
big_dataset.head()

Downloaded postal_codes: 34,868 rows × 62 columns
Downloaded rent: 34,900 rows × 13 columns
Downloaded income: 34,926 rows × 57 columns
Downloaded libraries: 15,704 rows × 46 columns
Downloaded bookshops: 3,237 rows × 8 columns
Downloaded population_history: 34,995 rows × 16 columns
rent: postalcode derived from INSEE_C; 52 unmatched rows kept.
income: postalcode derived from Code géog; 140 unmatched rows kept.
population_history: postalcode derived from codgeo; 155 unmatched rows kept.
Created big_merged_dataset.csv: 34,868 rows × 190 columns


,code_insee,nom_standard,nom_sans_pronom,nom_a,nom_de,nom_sans_accent,nom_standard_majuscule,typecom,typecom_texte,reg_code,...,bookshops_Commune,bookshops_code_commune,bookshops_Departement,bookshops_Region,population_history_reg,population_history_dep,population_history_cv,population_history_codgeo,population_history_libgeo,population_history_p23_pop
0,01001,L'Abergement-Clémenciat,Abergement-Clémenciat,à l'Abergement-Clémenciat,de l'Abergement-Clémenciat,l-abergement-clemenciat,L'ABERGEMENT-CLÉMENCIAT,COM,commune,84,...,CHATILLON SUR CHALARONNE,01093,AIN,AUVERGNE RHONE ALPES,82.0,01,0108,01393,Sandrans,582.0
1,01002,L'Abergement-de-Varey,Abergement-de-Varey,à l'Abergement-de-Varey,de l'Abergement-de-Varey,l-abergement-de-varey,L'ABERGEMENT-DE-VAREY,COM,commune,84,...,NaN,NaN,NaN,NaN,82.0,01,0101,01002,L' Abergement-de-Varey,270.0
2,01004,Ambérieu-en-Bugey,Ambérieu-en-Bugey,à Ambérieu-en-Bugey,d'Ambérieu-en-Bugey,amberieu-en-bugey,AMBÉRIEU-EN-BUGEY,COM,commune,84,...,AMBERIEU EN BUGEY,01004,AIN,AUVERGNE RHONE ALPES,82.0,01,0101,01007,Ambronay,2841.0
3,01005,Ambérieux-en-Dombes,Ambérieux-en-Dombes,à Ambérieux-en-Dombes,d'Ambérieux-en-Dombes,amberieux-en-dombes,AMBÉRIEUX-EN-DOMBES,COM,commune,84,...,VILLARS LES DOMBES,01443,AIN,AUVERGNE RHONE ALPES,82.0,01,0122,01382,Sainte-Olive,357.0
4,01006,Ambléon,Ambléon,à Ambléon,d'Ambléon,ambleon,AMBLÉON,COM,commune,84,...,BELLEY,01034,AIN,AUVERGNE RHONE ALPES,82.0,01,0104,01058,Brégnier-Cordon,804.0


Dropping useless columns

In [ ]:
safe_to_drop = [
    "url_wikipedia"

    # Rarely useful address / metadata fields
    "libraries_Cedex",
    "libraries_Complément",
    "libraries_libelle_2",
    "bookshops_label",

    # Duplicate geographical labels/identifiers already held in base dataset
    "rent_INSEE_C",
    "rent_LIBGEO",
    "rent_DEP",
    "rent_REG",
    "income_Nom géogr",
    "income_Code géog",
    "income_Libellé g",
    "libraries_code_region",
    "libraries_Région",
    "libraries_code_departement",
    "libraries_Département",
    "libraries_Ville",
    "libraries_Cedex",
    "libraries_code_insee_commune",
    "libraries_Population commune",
    "libraries_Longitude",
    "libraries_Latitude",
    "bookshops_Commune",
    "bookshops_code_commune",
    "bookshops_Departement",
    "bookshops_Region",
]

big_dataset = big_dataset.drop(columns=safe_to_drop, errors="ignore")

In [19]:
rows_with_missing = big_dataset.isna().any(axis=1).sum()

rows_without_missing = big_dataset.notna().all(axis=1).sum()

print(f"Rows with at least one missing value: {rows_with_missing:,}")
print(f"Rows with no missing values: {rows_without_missing:,}")

Rows with at least one missing value: 32,238
Rows with no missing values: 2,630


In [14]:
for col in big_dataset.columns:
    print(col)

code_insee
nom_standard
nom_sans_pronom
nom_a
nom_de
nom_sans_accent
nom_standard_majuscule
typecom
typecom_texte
reg_code
reg_nom
dep_code
dep_nom
canton_code
canton_nom
epci_code
epci_nom
postalcode
codes_postaux
academie_code
academie_nom
zone_emploi
zone_emploi_nom
code_insee_centre_zone_emploi
code_unite_urbaine
nom_unite_urbaine
taille_unite_urbaine
type_commune_unite_urbaine
statut_commune_unite_urbaine
code_aire_attraction
nom_aire_attraction
categorie_aire_attraction
taille_aire_attraction
code_insee_centre_aire_attraction
code_bassin_de_vie
nom_bassin_de_vie
type_bassin_de_vie
population
superficie_hectare
superficie_km2
densite
superficie_cartographique_ign_hectare
superficie_cartographique_ign_km2
densite_cartographique
superficie_cadastrale_ign_hectare
superficie_cadastrale_ign_km2
densite_cadastrale
altitude_moyenne
altitude_minimale
altitude_maximale
latitude_mairie
longitude_mairie
latitude_centre
longitude_centre
grille_densite
grille_densite_texte
niveau_equipements_s